# Activity 3 - JoPACC Reports Question-and-Answer System

This notebook follows the supplied RAG example. It uses pypdf extraction with a simple OCR fallback, overlapping chunks, sentence-transformer embeddings, cosine similarity, retrieved evidence, and an authorised language model.

In [ ]:
# This installs the packages we need. r
%pip install requests pypdf sentence-transformers rapidocr-onnxruntime pypdfium2

In [ ]:
# Here we bring in the libraries and set up the two things we need before anything else.
import os
import re
import numpy as np
import requests

import pypdfium2 as pdfium
from pypdf import PdfReader
from rapidocr_onnxruntime import RapidOCR
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# The OCR engine is created once here, because loading it takes a while and we reuse it for
# every scanned page.
ocr_engine = RapidOCR()

# These are the settings for the model we send the question to. 
# The key is read from the environment so it never lives in the notebook.
openrouter_api_key = os.environ.get("OPENROUTER_API_KEY", "")
openrouter_model = "minimax/minimax-m3:free"
openrouter_url = "https://openrouter.ai/api/v1/chat/completions"

In [ ]:
# This is the folder that holds the monthly reports.
folder_path = "../Reports"

# We use this to turn a month name that we find in a filename into its number.
month_numbers = {
    "january": 1, "february": 2, "march": 3,
    "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9,
    "october": 10, "november": 11, "december": 12
}

# clean_text removes the odd characters that come out of PDF extraction and turns any run of whitespace into one single space, so something like "Hello   world" becomes "Hello world".
def clean_text(text):
    text = text.replace("\x00", " ").replace("\x7f", " ")
    return re.sub(r"\s+", " ", text).strip()

# ocr_page isused if it draws the page as an image and then reads the text off that image,
# which is what we need for reports that were scanned instead of typed.
def ocr_page(pdf_document, page_index):
    image = pdf_document[page_index].render(scale=1.5).to_numpy()
    result, _ = ocr_engine(image)
    return clean_text(" ".join(line[1] for line in (result or [])))

# get_report_month works out which reporting period a file belongs to by looking for a month name
# and a year inside the filename. We need this because every passage has to know where it came from.
def get_report_month(filename):
    lower_name = filename.lower()
    month_name = next((name for name in month_numbers if name in lower_name), None)
    year_match = re.search(r"(20\d{2})", lower_name)

    if month_name and year_match:
        year = year_match.group(1)
        return f"{year}-{month_numbers[month_name]:02d}"

    return "Unknown"

In [ ]:
# Now we go through every report in the folder and pull the text out of it, one page at a time.
all_texts = []
ocr_reports = []

# Keep the example's pypdf method first. Use OCR only when a report
# contains too little extractable text.
minimum_report_characters = 1000

for filename in sorted(os.listdir(folder_path)):
    if filename.lower().endswith(".pdf"):
        path = os.path.join(folder_path, filename)
        reader = PdfReader(path)
        page_texts = [clean_text(page.extract_text() or "") for page in reader.pages]

        # If the whole report came back with almost no text, it is most likely a scanned document, so we do the same report again using OCR.
        if sum(len(text) for text in page_texts) < minimum_report_characters:
            print("Using OCR:", filename)
            ocr_document = pdfium.PdfDocument(path)
            page_texts = [ocr_page(ocr_document, index) for index in range(len(ocr_document))]
            ocr_document.close()
            ocr_reports.append(filename)

        report_month = get_report_month(filename)

        # We store every page on its own, together with the filename, the month and the page number.
        for page_number, page_text in enumerate(page_texts, start=1):
            if page_text:
                all_texts.append({
                    "source": filename,
                    "month": report_month,
                    "page": page_number,
                    "text": page_text
                })

print("Reports used:", len(set(item["source"] for item in all_texts)))
print("Extracted pages:", len(all_texts))
print("Reports processed with OCR:", len(ocr_reports))

In [ ]:
# A full page is too long to retrieve from, so we cut the text into smaller passages.
# Each chunk is 200 words, and we step back 40 words each time so the chunks overlap.
def chunk_text(text, max_words=200, overlap=40):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap

    return chunks

# Then we chunk every page and carry the same information over to each chunk.
all_chunks = []

for page in all_texts:
    chunks = chunk_text(page["text"])

    for chunk in chunks:
        all_chunks.append({
            "source": page["source"],
            "month": page["month"],
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(all_chunks))

In [ ]:
# This is the embedding model. It turns each chunk into a vector that represents its meaning,
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts)

In [ ]:
# retrieve_chunks is the search step. We encode the question the same way we encoded the chunks,
# then look for the passages that point in the closest direction to it.
def retrieve_chunks(question, top_k=5):
    query_emb = embed_model.encode([question])
    periods = requested_months(question)

    # If the question names a reporting period, search only those reports.
    candidate_indices = [
        index for index, chunk in enumerate(all_chunks)
        if not periods or chunk["month"] in periods
    ]

    # If a requested period is unavailable, retrieve general evidence and
    # let the prompt explain that the requested answer was not found.
    if not candidate_indices:
        candidate_indices = list(range(len(all_chunks)))

    # Cosine similarity gives us a score for every candidate, then we keep the highest ones.
    candidate_embeddings = chunk_embeddings[candidate_indices]
    similarities = cosine_similarity(query_emb, candidate_embeddings)[0]
    local_top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    # We return each passage together with its score and its source, because the application
    # displays both of them next to the answer.
    for local_index in local_top_indices:
        index = candidate_indices[local_index]
        results.append({
            "score": float(similarities[local_index]),
            "source": all_chunks[index]["source"],
            "month": all_chunks[index]["month"],
            "page": all_chunks[index]["page"],
            "text": all_chunks[index]["text"]
        })

    return results

In [ ]:
# Anything below this score is treated as too weak to be used as evidence.
minimum_relevance_score = 0.45

month_pattern = "|".join(month_numbers)

# requested_months looks for something like "June 2026" inside the question and turns it into
# 2026-06, which is the same format we gave the chunks. This is what lets the search above filter
# by period. It is written after retrieve_chunks, but that is fine because it only gets called
# when the search actually runs.
def requested_months(question):
    matches = re.findall(
        rf"\b({month_pattern})\s+(20\d{{2}})\b",
        question.lower()
    )
    return {f"{year}-{month_numbers[month_name]:02d}" for month_name, year in matches}

In [ ]:
# rag_query does everything we talked about 
def rag_query(question, top_k=5):
    retrieved = retrieve_chunks(question, top_k)

    # If nothing came back strong enough, we stop here and do not ask the model at all.
    if not retrieved or retrieved[0]["score"] < minimum_relevance_score:
        return "Not found in the supplied JoPACC reports.", retrieved

    # We paste the retrieved passages into one block of context, and we keep the source, month and
    # page on each one so the model can tell the different reports apart.
    context = ""

    for result in retrieved:
        context += (
            f"\n\nSOURCE: {result['source']}"
            f" | MONTH: {result['month']}"
            f" | PAGE: {result['page']}\n"
            f"{result['text']}"
        )

    # The prompt tells the model to use only this context, to ask for a month when the question does
    # not give one, and to say the answer was not found instead of guessing at it.
    prompt = f"""
Use ONLY the information in the context to answer the question.

Before answering, decide whether the question is sufficiently specific.
- If a monthly number, value, total or average is requested without a month and year, ask the user to provide them.
- If an average is ambiguous, ask which payment system, reporting period and type of average is required.
- Treat CliQ as the instant-payment system for transfers issued by banks, and JoMoPay as transfers issued by mobile wallets.
- If the question is unrelated to JoPACC payments, or the requested answer is not supported by the context, reply: "Not found in the supplied JoPACC reports."
- Do not guess missing details.

Write the answer in 2-3 short paragraphs with:
- clear and concise wording
- no unnecessary details
- no external knowledge
- no invented values

If the context contains different values from different months, clearly identify each month.

Context:
{context}

Question: {question}

Answer:
"""

    # Then we send it. We use a low temperature because we want the same answer for the same
    # question rather than a creative one.
    response = requests.post(
        openrouter_url,
        headers={
            "Authorization": f"Bearer {openrouter_api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": openrouter_model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
        },
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]

    return answer.strip(), retrieved

In [ ]:
# This prints the answer and then the passages it was based on, so we can check the report,
# the month, the page and the score behind every answer we get.
def display_result(question, answer, documents):
    print("QUESTION:", question)
    print("ANSWER:", answer)

    for document in documents:
        print("\n--- Supporting evidence ---")
        print("Report:", document["source"])
        print("Month:", document["month"])
        print("Page:", document["page"])
        print("Relevance score:", round(document["score"], 3))
        print("Passage:", document["text"])

In [ ]:
# And here we run one example question to check that the whole thing works from end to end.
question = "What was CliQ's transaction value in June 2026?"

answer, documents = rag_query(question)
display_result(question, answer, documents)